In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# Veriyi yükleme
df = pd.read_csv('/content/KoreanDataset_5.0.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Date_ordinal'] = df['Date'].map(lambda x: x.toordinal())

# Özellikler ve ürünler
features = ['Date_ordinal', 'AveragePrice', 'WTIPrice', 'BrentPrice', 'tmax', 'tmin', 'tavg', 'wspd', 'pres']
products = ['Latte Varieties', 'Sweets and Cakes', 'Croissant Varieties']

# Veriyi ölçeklendirme
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(df[features].values)

def create_dataset(X, y, window_size=35, group_size=7):
    X_data, y_data = [], []
    for i in range(len(X) - window_size - group_size + 1):
        X_window = X[i:i + window_size].reshape(-1, group_size, X.shape[1])
        X_avg = np.mean(X_window, axis=1)
        y_avg = np.mean(y[i + window_size:i + window_size + group_size])
        X_data.append(X_avg)
        y_data.append(y_avg)
    return np.array(X_data), np.array(y_data)

results = {}

for product in products:
    print(f"Eğitilen ürün: {product}")
    y = df[product].values
    X_window, y_window = create_dataset(X_scaled, y)

    X_train, X_test, y_train, y_test = train_test_split(X_window, y_window, test_size=0.2, shuffle=True)

    model = Sequential([
        Bidirectional(LSTM(64, return_sequences=True, recurrent_dropout=0.2), input_shape=(X_train.shape[1], X_train.shape[2])),
        BatchNormalization(),
        Bidirectional(LSTM(32, return_sequences=False, recurrent_dropout=0.2)),
        Dense(16, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='linear')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=1000, batch_size=32, validation_data=(X_test, y_test), verbose=1)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[product] = {'RMSE': rmse, 'MSE': mse, 'MAE': mae, 'R2': r2}

    # Test ve tahmin edilen değerleri txt'ye yaz
    with open(f"test_vs_pred_{product.replace(' ', '_')}.txt", "w") as file:
        file.write(f"Ürün: {product}\n")
        file.write("Gerçek Değerler (y_test) - Tahmin Değerleri (y_pred)\n")
        for real, pred in zip(y_test, y_pred.flatten()):
            file.write(f"{real:.4f} - {pred:.4f}\n")

full_feature_df = pd.DataFrame(results).T

print("\nModel Performans Sonuçları:")
print(full_feature_df)

fig, axes = plt.subplots(4, 1, figsize=(10, 20))
metrics = ['RMSE', 'MSE', 'MAE', 'R2']

for i, metric in enumerate(metrics):
    axes[i].bar(full_feature_df.index, full_feature_df[metric], width=0.4, label='Tüm Featurelar', align='edge')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_xlabel('Ürünler')
    axes[i].set_ylabel(metric)
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# Veriyi yükleme
df = pd.read_csv('/content/KoreanDataset_5.0.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Date_ordinal'] = df['Date'].map(lambda x: x.toordinal())

# Özellikler ve ürünler
features = ['Date_ordinal', 'AveragePrice', 'WTIPrice', 'BrentPrice', 'tmax', 'tmin', 'tavg', 'wspd', 'pres']
products = ['Fruity and Refreshing Drinks', 'Snacks', 'Classic Breads and Toasts']

# Veriyi ölçeklendirme
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(df[features].values)

def create_dataset(X, y, window_size=35, group_size=7):
    X_data, y_data = [], []
    for i in range(len(X) - window_size - group_size + 1):
        X_window = X[i:i + window_size].reshape(-1, group_size, X.shape[1])
        X_avg = np.mean(X_window, axis=1)
        y_avg = np.mean(y[i + window_size:i + window_size + group_size])
        X_data.append(X_avg)
        y_data.append(y_avg)
    return np.array(X_data), np.array(y_data)

results = {}

for product in products:
    print(f"Eğitilen ürün: {product}")
    y = df[product].values
    X_window, y_window = create_dataset(X_scaled, y)

    X_train, X_test, y_train, y_test = train_test_split(X_window, y_window, test_size=0.2, shuffle=True)

    model = Sequential([
        Bidirectional(LSTM(64, return_sequences=True, recurrent_dropout=0.2), input_shape=(X_train.shape[1], X_train.shape[2])),
        BatchNormalization(),
        Bidirectional(LSTM(32, return_sequences=False, recurrent_dropout=0.2)),
        Dense(16, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='linear')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
    model.fit(X_train, y_train, epochs=1000, batch_size=32, validation_data=(X_test, y_test), verbose=1)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[product] = {'RMSE': rmse, 'MSE': mse, 'MAE': mae, 'R2': r2}

    # Test ve tahmin edilen değerleri txt'ye yaz
    with open(f"test_vs_pred_{product.replace(' ', '_')}.txt", "w") as file:
        file.write(f"Ürün: {product}\n")
        file.write("Gerçek Değerler (y_test) - Tahmin Değerleri (y_pred)\n")
        for real, pred in zip(y_test, y_pred.flatten()):
            file.write(f"{real:.4f} - {pred:.4f}\n")

full_feature_df = pd.DataFrame(results).T

print("\nModel Performans Sonuçları:")
print(full_feature_df)

fig, axes = plt.subplots(4, 1, figsize=(10, 20))
metrics = ['RMSE', 'MSE', 'MAE', 'R2']

for i, metric in enumerate(metrics):
    axes[i].bar(full_feature_df.index, full_feature_df[metric], width=0.4, label='Tüm Featurelar', align='edge')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_xlabel('Ürünler')
    axes[i].set_ylabel(metric)
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
"""
from google.colab import drive
drive.mount('/content/drive')
"""